# 06. 서버 등록(가입) — 관리 서버 자신 · 다른 서버 · 재가입

| 항목 | 내용 |
|---|---|
| 목표 | 서버에 에이전트를 설치해 사이트에 등록하고, 관리자 계정·SSH 인증서 설정을 적용 |
| 소요 시간 | 서버당 15~20분 |
| 시작 전 조건 | 05 완료(승인된 서버 관리자), 대상 서버가 사설 IP 로 전환됨(01) |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## 가입이 하는 일 (서버 한 대 기준)
1. 사이트 **서버 추가·관리**에서 서버 이름·사설 IP 를 등록 → **서버에서 실행할 한 줄 명령**이 나옵니다(30분 유효, **1회용**).
2. 서버에서 명령을 실행하면 스크립트가: 사설 IP 확인 → 에이전트(`ssm-agent`) 설치·시작(사설 IP 에만 바인딩, 자체 서명 인증서 + 토큰) → 방화벽 **허용** 규칙 → 로그인 안내문(MOTD) → 셸 연동 파일 → 사이트에 등록(토큰 소모) → 재부팅 전용 SSH 키 설치.
3. 사이트가 에이전트에게 **SSH 인증서 신뢰 설정**(`TrustedUserCAKeys`)을 넣고, 승인된 **관리자 계정(sudo)** 을 만듭니다.
4. 서버 카드가 **온라인**으로 바뀌고 웹 터미널로 접속됩니다.

![가입 순서](img/join.svg)

### 가입 토큰
- **16글자(0~9, a~f)** 입니다. 사이트에는 4글자씩 끊어 크게 보여 줍니다(복사는 띄어쓰기 없는 값).
- **명령 재발급**을 누르면 이전 토큰은 무효가 됩니다. 새 토큰이 나왔다는 알림(시각·몇 번째)이 뜹니다.

## A. 사이트에서 서버 등록 (🌐)
**서버 추가·관리** → "새 서버": 이름(`[MGMT_NAME]`), 사설 IP, 관리 서버면 **관리 서버(학생 sudo 없음)** 체크 → 등록 → 나온 명령을 복사(또는 토큰 확인).

## B. 일반 서버(관리 서버가 아닌 서버) — 서버 콘솔에서

사이트가 보여 주는 명령을 그대로 실행합니다. 형태는 아래와 같습니다. 서버가 인터넷에서 `[FUNNEL_URL]` 에 닿아야 합니다.

In [ ]:
!cd ~
!curl -sI -m 10 [FUNNEL_URL]/                                   # HTTP/2 200 이면 사이트에 닿음
!curl -fsSL [FUNNEL_URL]/api/join/script -o join-node.sh && sudo SSM_URL=[FUNNEL_URL] SSM_MGMT_IP=[MGMT_PRIVATE_IP] SSM_JOIN_TOKEN=[TOKEN_16] bash join-node.sh [SERVER_NAME]

마지막에 이렇게 나오면 성공입니다.
```
== Done: [SERVER_NAME] ([PRIVATE_IP]) is registered with the site.
```
그 뒤에 `!! some follow-up steps failed: …` 줄이 **없어야** 합니다(있으면 사이트 **감사 로그**의 `server_join_partial` 에서 어떤 항목이 실패했는지 확인).

## C. 관리 서버 **자신**의 가입 — 자기 Funnel 주소 우회

**증상**: 관리 서버에서 자기 사이트 주소로 접속하면 `curl: (60) SSL certificate problem: self-signed certificate`.
**원인**: 같은 서버 안에서는 그 주소가 Tailscale 이 아니라 K3s 웹 입구(Traefik)의 **임시 자체 서명 인증서**로 연결됩니다. (외부와 다른 서버는 정상 인증서를 받습니다.)
**우회**: 서버가 자기 사이트에 **인터넷 경로(Funnel)로** 접속하도록 `/etc/hosts` 에 **임시로** 매핑하고, 끝나면 지웁니다.

In [ ]:
!sudo apt install -y dnsutils
!dig +short @1.1.1.1 [MGMT_NAME].[TAILNET].ts.net               # Funnel 공인 주소 한두 개가 나옴(첫 번째 사용)

In [ ]:
!echo "[FUNNEL_IP] [MGMT_NAME].[TAILNET].ts.net" | sudo tee -a /etc/hosts
!curl -sI -m 10 [FUNNEL_URL]/                                   # HTTP/2 200 (10초 안에 안 되면 다른 주소로 교체)
!cd ~
# → 사이트의 등록(재가입) 명령을 그대로 실행
!sudo sed -i '/[MGMT_NAME].[TAILNET].ts.net/d' /etc/hosts       # 성공이든 실패든 꼭 지운다

> `curl` 로 확인할 때 `http://` 로 치면 Funnel 이 `https://` 로 돌려보내는 **302** 가 나옵니다(정상). `https://` 로 쳐야 `200` 입니다.
> `-o`(영문 오)를 `-i`(아이)로 잘못 치면 `000` 이 나옵니다.

## D. 가입 후 확인
- 사이트 **서버 현황**에서 서버 카드가 **온라인**이고 GPU 모델·VRAM 이 보임.
- 서버에서:

In [ ]:
!systemctl is-active ssm-agent                  # active
!sudo ss -ltnp | grep -E ':7541|sshd'            # 에이전트는 사설 IP 의 7541, sshd 는 [SSH_PORT]
!ls /etc/ssh/sshd_config.d/                      # 50-ssm-ca.conf 가 보여야 웹 터미널이 접속됨
!getent passwd [ADMIN_ID]                        # 승인된 관리자 계정이 만들어졌는지

## E. 재가입 (이미 등록된 서버를 다시 설정)
에이전트 재설치나 SSH 설정·관리자 계정 **재적용**이 필요할 때, **서버 추가·관리의 "등록된 서버" 표**에서 해당 행의 **재가입 명령**을 누르면 새 16글자 토큰이 나옵니다(상태가 정상이어도 됩니다). 이후 B 또는 C 와 같은 방법으로 실행하세요. 위쪽 "새 서버" 입력칸에 같은 이름을 넣으면 "이미 등록된 서버"가 나오니 쓰지 마세요.

## F. 이미 사용 중인 서버(기존 학생 계정이 있음)
가입한 뒤 **팀 관리 → 기존 서버 계정 가져오기**에서 서버·계정·팀 정보·사용 일수를 넣으면 **설정 링크(24시간·1회용)** 가 나옵니다. 그 팀에게만 전달 → 팀이 새 비밀번호를 정하면 사이트와 서버 계정이 연결됩니다. 서버 관리자 개인 계정은 가져오지 말고 사이트에서 관리자로 가입·승인받으세요.

## G. 제거(서버에서 되돌리기)

In [ ]:
!sudo systemctl disable --now ssm-agent
!sudo rm -rf /opt/ssm-agent /etc/ssm-agent /etc/ssm /etc/update-motd.d/90-ssm-expiry /etc/profile.d/ssm-osc7.sh
# /root/.ssh/authorized_keys 의 ssm-reboot-key 줄, /etc/ssh/sshd_config.d/50-ssm-ca.conf, 61-ssm-root-forced.conf 도 지우고 sshd 를 재적재
# 사이트에서는 서버를 점검 상태로 바꿉니다

## 문제가 생기면
| 증상 | 원인·조치 |
|---|---|
| `could not download the agent package` | 토큰이 틀리거나 만료/재발급됨 → 새 토큰. 토큰만 검사하려면 `curl -s -o /dev/null -w '%{http_code}\n' "[FUNNEL_URL]/api/join/agent.tar.gz?name=[SERVER_NAME]&token=$TOKEN"` (`200` 정상, `403` 토큰 문제) |
| `ssm-agent did not start`, `start-limit-hit` | 서비스가 반복 실패해 systemd 가 막음 → 원인 수정 후 `sudo systemctl reset-failed ssm-agent` |
| `ssh_ca: agent_error` + 계정이 안 만들어짐 | 에이전트가 `/etc` 에 쓰지 못함(서비스의 `ProtectSystem`) → 최신 스크립트로 재가입, 또는 임시로 덮어쓰기 파일(08 참고) |
| `cert_ip_mismatch` | 사이트에 등록한 사설 IP 와 서버의 실제 IP 가 다름 |
| 웹 터미널 "서버에 접속하지 못했습니다" | SSH 포트·계정 없음·CA 신뢰 설정 중 하나. 서버에서 `ls /etc/ssh/sshd_config.d/`, `getent passwd`, `sudo journalctl -u ssh -n 15 --no-pager` |

## ✅ 완료 확인

- [ ] 서버 카드가 온라인이다
- [ ] `50-ssm-ca.conf` 가 있고 가입 마지막에 `follow-up steps failed` 가 없다
- [ ] 관리자로 웹 터미널에 접속된다
- [ ] 관리 서버라면 `/etc/hosts` 의 임시 줄을 지웠다